In [14]:
import polars as pl

pl.Config.set_tbl_cols(-1)      # Show all columns
pl.Config.set_tbl_rows(20)      # Optional: show more rows
pl.Config.set_tbl_width_chars(800)  # Increase table width

polars.config.Config

In [15]:
import polars as pl

routes = [
    "M1",
    "M2",
    "M4",
    "M15",
    "M101"
]

url = (
    "http://parquet.gtfsrt.io/vehicle_positions"
    "/date=2026-07-01"
    "/base64url=aHR0cHM6Ly9ndGZzcnQucHJvZC5vYmFueWMuY29tL3ZlaGljbGVQb3NpdGlvbnM"
    "/data.parquet"
)

df = (
    pl.scan_parquet(url)
      .filter(pl.col("route_id").is_in(routes))
      .select([
      "trip_id",
      "route_id",
      "vehicle_id",
      "direction_id",
    #   "start_time",
      "start_date",
    #   "vehicle_label",
      "latitude",
      "longitude",
      "bearing",
      "stop_id",
      "timestamp"
  ])

      .collect(engine="streaming")
)

print(df.shape)
print(df.head())

(213659, 10)
shape: (5, 10)
┌─────────────────────────────┬──────────┬───────────────┬──────────────┬────────────┬───────────┬────────────┬────────────┬─────────┬────────────┐
│ trip_id                     ┆ route_id ┆ vehicle_id    ┆ direction_id ┆ start_date ┆ latitude  ┆ longitude  ┆ bearing    ┆ stop_id ┆ timestamp  │
│ ---                         ┆ ---      ┆ ---           ┆ ---          ┆ ---        ┆ ---       ┆ ---        ┆ ---        ┆ ---     ┆ ---        │
│ str                         ┆ str      ┆ str           ┆ u32          ┆ str        ┆ f32       ┆ f32        ┆ f32        ┆ str     ┆ u64        │
╞═════════════════════════════╪══════════╪═══════════════╪══════════════╪════════════╪═══════════╪════════════╪════════════╪═════════╪════════════╡
│ MV_C6-Weekday-119400_M4_434 ┆ M4       ┆ MTA NYCT_9771 ┆ 1            ┆ 20260630   ┆ 40.845314 ┆ -73.940529 ┆ 247.760437 ┆ 400648  ┆ 1782863966 │
│ MV_C6-Weekday-114100_M4_433 ┆ M4       ┆ MTA NYCT_9775 ┆ 0            ┆ 20260630  

In [16]:
df.null_count()


trip_id,route_id,vehicle_id,direction_id,start_date,latitude,longitude,bearing,stop_id,timestamp
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


In [17]:
df.write_parquet(
    "raw/vehicle_positions/2026-07-01.parquet"
)

In [18]:
import polars as pl

routes = [
    "M1",
    "M2",
    "M4",
    "M15",
    "M101"
]

url = (
    "http://parquet.gtfsrt.io/trip_updates"
    "/date=2026-07-01"
    "/base64url=aHR0cHM6Ly9ndGZzcnQucHJvZC5vYmFueWMuY29tL3RyaXBVcGRhdGVz"
    "/data.parquet"
)

df = (
    pl.scan_parquet(url)
      .filter(pl.col("route_id").is_in(routes))
      .select([
        "feed_timestamp",
        "fetch_timestamp",
        "trip_id",
        "route_id",
        "direction_id",
        "start_time",
        "start_date",
        "schedule_relationship",
        "vehicle_id",
        "trip_timestamp",
        "stop_sequence",
        "stop_id",
        "arrival_time",
        "departure_time"
  ])

      .collect(engine="streaming")
)

print(df.shape)
print(df.head())

(8209429, 14)
shape: (5, 14)
┌────────────────┬────────────────────────────────┬─────────────────────────────┬──────────┬──────────────┬────────────┬────────────┬───────────────────────┬───────────────┬────────────────┬───────────────┬─────────┬──────────────┬────────────────┐
│ feed_timestamp ┆ fetch_timestamp                ┆ trip_id                     ┆ route_id ┆ direction_id ┆ start_time ┆ start_date ┆ schedule_relationship ┆ vehicle_id    ┆ trip_timestamp ┆ stop_sequence ┆ stop_id ┆ arrival_time ┆ departure_time │
│ ---            ┆ ---                            ┆ ---                         ┆ ---      ┆ ---          ┆ ---        ┆ ---        ┆ ---                   ┆ ---           ┆ ---            ┆ ---           ┆ ---     ┆ ---          ┆ ---            │
│ u64            ┆ datetime[μs, UTC]              ┆ str                         ┆ str      ┆ u32          ┆ str        ┆ str        ┆ i32                   ┆ str           ┆ u64            ┆ u32           ┆ str     ┆ i64    

In [19]:
df.write_parquet(
    "raw/trip_updates/2026-07-01.parquet"
)

In [20]:
import polars as pl

url = (
    "http://parquet.gtfsrt.io/service_alerts"
    "/date=2026-07-01"
    "/base64url=aHR0cHM6Ly9ndGZzcnQucHJvZC5vYmFueWMuY29tL2FsZXJ0cw"
    "/data.parquet"
)

df = (
    pl.scan_parquet(url)
    .select([
        "feed_timestamp",
        "fetch_timestamp",
        "entity_id",
        "active_period_start",
        "active_period_end",
        "header_text",
        "description_text",
        "url",
        "route_id",
        "route_type",
        "stop_id",
        "trip_id",
        "trip_route_id",
        "trip_direction_id"
    ])
    .collect(engine="streaming")
)

print(df.shape)
print(df.head())

(208877, 14)
shape: (5, 14)
┌────────────────┬────────────────────────────────┬─────────────────────────────────┬─────────────────────┬───────────────────┬─────────────────────────────────┬─────────────────────────────────┬──────┬──────────┬────────────┬─────────┬─────────┬───────────────┬───────────────────┐
│ feed_timestamp ┆ fetch_timestamp                ┆ entity_id                       ┆ active_period_start ┆ active_period_end ┆ header_text                     ┆ description_text                ┆ url  ┆ route_id ┆ route_type ┆ stop_id ┆ trip_id ┆ trip_route_id ┆ trip_direction_id │
│ ---            ┆ ---                            ┆ ---                             ┆ ---                 ┆ ---               ┆ ---                             ┆ ---                             ┆ ---  ┆ ---      ┆ ---        ┆ ---     ┆ ---     ┆ ---           ┆ ---               │
│ u64            ┆ datetime[μs, UTC]              ┆ str                             ┆ u64                 ┆ u64            

In [21]:
df.null_count()


feed_timestamp,fetch_timestamp,entity_id,active_period_start,active_period_end,header_text,description_text,url,route_id,route_type,stop_id,trip_id,trip_route_id,trip_direction_id
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,25622,0,0,208877,152314,208877,208877,56563,56563,56563


In [22]:
df.write_parquet(
    "raw/service_alerts/2026-07-01.parquet"
)